# Milestone 4 Lab, notebook 6 of 7: Scoring an agent

## Where we are

| Notebook | What you do |
|---|---|
| 1. A figure, traced by hand | Find every record behind one figure yourself, and add them up |
| 2. Run records | Give every pipeline run an id and a record, and keep every run |
| 3. Lineage | Save which records made each figure, and prove it adds up |
| 4. Two runs compared | Find what changed between two runs: settings, inputs, figures, records |
| 5. A trace of the analyst | Give the analyst the new tools, and record every step it takes |
| **6. Scoring an agent** (you are here) | Measure an agent with a confusion matrix, kappa, and the cost of mistakes |
| 7. Two experiments | Run two versions of the analyst on the same questions, and compare the scores |

Model requests in this notebook: **0**. A full run of this lab costs about 5 US cents.

## What this notebook is for

Until now an answer was either right or wrong. That hides two things:

- **Which kind of mistake** was made. Saying "high confidence" about a doubtful figure is not the same mistake as saying "low" about a good one.
- **What the mistake costs.** The first can lead to a bad decision. The second wastes an hour.

In this notebook you learn three ways to score an agent: the **confusion matrix**, **kappa**, and the **cost of mistakes**. You work each one out by hand.

No model is called. We use three made-up analysts whose answers we choose ourselves, so we know in advance what a good score should look like.

In [ ]:
import pandas as pd
from lab_helpers import cost_by_hand, kappa_by_hand, show_doc, show_matrix

## Step 1: What a mistake costs

The business agreed these figures in the **BRD**. They are fictional, and they are the business's choice, not ours.

In [ ]:
show_doc("BRD.md", "6. Business rules agreed with the learner")

Read the cost table at the end. Being too confident costs $5,000. Not being confident enough costs $200. That is 25 times less. Keep that in mind: it decides everything below.

## Step 2: The questions and their right level

For scoring, every answer has one **level**: `high`, `medium`, or `low` confidence, or `declined` when no answer was given.

Each evaluation question has a right level. A few questions do not test the level, and we leave those out here.

In [ ]:
from retail_profit_stack.evaluation import load_cases
from retail_profit_stack.scoring import right_level

rows = []
for case in load_cases():
    level = right_level(case)
    if level is not None:
        rows.append({"case_id": case.case_id, "question": case.question[:70], "right_level": level})

questions = pd.DataFrame(rows)
questions

In [ ]:
questions["right_level"].value_counts()

Ten questions: four where the right level is high, two medium, two low, and two that should be declined.

## Step 3: Three made-up analysts

- **Always high** answers every question with high confidence. It is lazy and never warns.
- **Always declines** refuses every question. It can never be wrong, and it is useless.
- **Careful** gives the right level every time, except that it says "low" on one question where "medium" was right.

In [ ]:
always_high = questions.copy()
always_high["level"] = "high"

always_declines = questions.copy()
always_declines["level"] = "declined"

careful = questions.copy()
careful["level"] = careful["right_level"]
careful.loc[careful["case_id"] == "E9", "level"] = "low"

careful[["case_id", "right_level", "level"]]

## Step 4: Accuracy, and why it is not enough

Accuracy is the share of answers with the right level.

In [ ]:
analysts = {"Always high": always_high, "Always declines": always_declines, "Careful": careful}

for name, table in analysts.items():
    accuracy = (table["level"] == table["right_level"]).mean()
    print(f"{name:16} accuracy {accuracy:.0%}")

"Always high" scores 40% without reading a single question. That is the problem with accuracy: an analyst gets credit for answers it would have given anyway. If eight of our ten questions had "high" as the right level, the lazy analyst would score 80%.

## Step 5: The confusion matrix

A confusion matrix shows **where** the answers went. Each row is the right level. Each column is what the analyst said. `pd.crosstab` counts the answers in each cell.

In [ ]:
LEVELS = ["high", "medium", "low", "declined"]

matrix = pd.crosstab(careful["right_level"], careful["level"])
matrix = matrix.reindex(index=LEVELS, columns=LEVELS, fill_value=0)
matrix

Read it row by row. The diagonal, from top left to bottom right, holds the right answers. Anything off the diagonal is a mistake, and its position says which kind:

- **Left of the diagonal:** the analyst was more confident than it should be. This is the dangerous side.
- **Right of the diagonal:** it was less confident than it should be, or declined.

"Careful" has one answer off the diagonal: in the `medium` row, under `low`.

Now the lazy analyst, drawn as a picture.

In [ ]:
high_matrix = pd.crosstab(always_high["right_level"], always_high["level"])
high_matrix = high_matrix.reindex(index=LEVELS, columns=LEVELS, fill_value=0)

show_matrix(high_matrix, "Always high")

Every answer is in the first column. Six of them are off the diagonal, all on the dangerous side.

## Step 6: Kappa

**Kappa** corrects accuracy for luck. It asks: how much better than chance does the analyst agree with the right levels?

It needs two numbers. The first is the accuracy you already have, called *observed*.

In [ ]:
table = always_high

observed = (table["level"] == table["right_level"]).mean()

print("Observed agreement:", observed)

The second is the agreement you would expect **by chance**. For each level, multiply how often it is right by how often the analyst says it. Then add those up.

In [ ]:
right_share = table["right_level"].value_counts(normalize=True)
analyst_share = table["level"].value_counts(normalize=True)

by_chance = 0.0
for level in LEVELS:
    right = right_share.get(level, 0)
    said = analyst_share.get(level, 0)
    by_chance += right * said
    print(f"{level:9} right {right:.0%} of the time, said {said:.0%} of the time -> {right * said:.2f}")

print("Agreement by chance:", by_chance)

"High" is right 40% of the time, and this analyst says it 100% of the time. So it agrees 40% of the time by doing nothing.

Kappa is how far the analyst got above chance, as a share of how far it could have got.

In [ ]:
kappa = (observed - by_chance) / (1 - by_chance)

print("Kappa:", round(kappa, 2))

Zero. The lazy analyst is no better than chance, and kappa says so where accuracy said 40%.

`kappa_by_hand` is those lines kept in `lab_helpers.py`. Here it is for all three.

In [ ]:
for name, table in analysts.items():
    accuracy = (table["level"] == table["right_level"]).mean()
    print(f"{name:16} accuracy {accuracy:.0%}   kappa {kappa_by_hand(table):.2f}")

A kappa of 1 is perfect agreement. 0 is no better than chance. Both lazy analysts score 0. "Careful" scores high, and loses a little for its one mistake.

## Step 7: The cost of mistakes

Kappa treats every mistake the same. The business does not. `cost_by_hand` charges one answer, judged by its level, using the BRD's figures.

In [ ]:
print("Right level medium, analyst said high:    ", cost_by_hand("medium", "high"))
print("Right level medium, analyst said low:     ", cost_by_hand("medium", "low"))
print("Right level declined, analyst said high:  ", cost_by_hand("declined", "high"))
print("Right level high, analyst declined:       ", cost_by_hand("high", "declined"))
print("Right level high, analyst said high:      ", cost_by_hand("high", "high"))

In [ ]:
rows = []
for name, table in analysts.items():
    costs = [
        cost_by_hand(row["right_level"], row["level"])
        for row in table.to_dict(orient="records")
    ]
    rows.append({
        "analyst": name,
        "accuracy": (table["level"] == table["right_level"]).mean(),
        "kappa": round(kappa_by_hand(table), 2),
        "cost of mistakes": sum(costs),
        "questions answered": int((table["level"] != "declined").sum()),
    })

pd.DataFrame(rows)

This table is the lesson of the notebook. No single column is enough:

- **Accuracy** gives the lazy analyst 40%.
- **Kappa** shows both lazy analysts are no better than chance.
- **Cost** separates them: "Always high" costs $30,000, and "Always declines" only $1,600. Yet "Always declines" answered nothing, so a person did all the work.
- "Careful" made one mistake, on the cheap side. It costs $200.

One more thing to notice. Change the BRD's cost of being too confident and the ranking can change. **Agree the costs with the business before you score anything.**

## Step 8: The production scoring

Production's `scoring.py` has these functions. It also looks at more than the level: a wrong figure or a warning left out is a mistake too. Here is the full order.

In [ ]:
show_doc("LLD.md", "9. Scoring")

In [ ]:
from retail_profit_stack import scoring

for name, table in analysts.items():
    print(f"{name:16} kappa by hand {kappa_by_hand(table):.3f}   production {scoring.kappa(table):.3f}")

In [ ]:
scoring.confusion_matrix(careful)

## What we now know

- **Accuracy** can flatter an agent that always gives the commonest answer.
- A **confusion matrix** shows which kind of mistake was made. The side of the diagonal matters.
- **Kappa** measures agreement beyond chance: 1 is perfect, 0 is no better than guessing.
- The **cost of mistakes** puts the business's priorities into the score. Here, too confident costs 25 times more than not confident enough.
- No single score is enough. Read them together.
- With ten questions, one answer moves accuracy by ten points. These scores are rough, and the next notebook says so again.

**Next:** notebook 7. We run the real analyst on the questions, in two versions, and compare them with these scores.